# Week 2 — Exploratory Data Analysis (EDA)
## LendingClub Loan Dataset (Finance Domain)

**Objective.** Conduct a comprehensive EDA on the LendingClub cleaned dataset
produced in Week 1, uncovering patterns, trends and relationships between
variables using descriptive statistics, visualization and hypothesis testing.

**Dataset.** The Week-1 cleaned dataset — 42,535 loans × 55 columns — built
from the publicly available LendingClub LoanStats3a export.

**Pipeline sections.**
1. Load cleaned dataset
2. Descriptive statistics (central tendency, dispersion, skew, kurtosis)
3. Categorical summary
4. Target variable analysis (loan_status → is_default)
5. Univariate visualizations (histograms, boxplots, QQ plots)
6. Bivariate analysis (scatter, correlation, group comparisons)
7. Multivariate analysis (correlation matrix, pair plot, heatmap)
8. Hypothesis testing (t-test, ANOVA, chi-square, Pearson)
9. Business insights and inferences

---
> **Run instructions.** Upload this notebook to Google Colab and run cells
> top-to-bottom (Runtime → Run all). It assumes `cleaned_dataset.csv` from
> Week 1 is in the working directory — upload it first, or rerun the Week-1
> notebook to regenerate it.

## 0. Environment setup

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 130

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f"pandas {pd.__version__}, numpy {np.__version__}")


## 1. Load the cleaned dataset

In [ ]:
CSV_PATH = "cleaned_dataset.csv"  # from Week 1
df = pd.read_csv(CSV_PATH)

# Restore datetime parsing
for c in ["issue_d", "earliest_cr_line", "last_pymnt_d", "next_pymnt_d", "last_credit_pull_d"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Numeric: {df.select_dtypes(include=[np.number]).shape[1]} | "
      f"Categorical: {df.select_dtypes(include=['object']).shape[1]} | "
      f"Datetime: {df.select_dtypes(include=['datetime64']).shape[1]}")
df.head(3)


## 2. Descriptive statistics

We compute central tendency (mean, median), dispersion (std, variance, IQR,
MAD, coefficient of variation), and shape (skew, kurtosis) for the key
numeric features.

In [ ]:
KEY_NUMERIC = ["loan_amnt","annual_inc","int_rate","installment","dti",
                "revol_bal","revol_util","total_acc","open_acc",
                "credit_history_years","loan_to_income","installment_to_income",
                "delinq_2yrs","inq_last_6mths","pub_rec"]

rows = []
for c in KEY_NUMERIC:
    s = pd.to_numeric(df[c], errors="coerce").dropna()
    rows.append({
        "feature": c, "n": int(s.count()),
        "mean": s.mean(), "median": s.median(),
        "std": s.std(), "var": s.var(),
        "min": s.min(), "max": s.max(), "range": s.max()-s.min(),
        "q1": s.quantile(0.25), "q3": s.quantile(0.75),
        "iqr": s.quantile(0.75)-s.quantile(0.25),
        "skew": s.skew(), "kurtosis": s.kurt(),
        "cv": s.std()/s.mean() if s.mean() else np.nan,
        "mad": (s - s.mean()).abs().mean(),
    })
desc = pd.DataFrame(rows).set_index("feature")
desc.round(3)


## 3. Categorical summary

In [ ]:
CAT_COLS = ["grade","sub_grade","home_ownership","verification_status",
            "loan_status","purpose","addr_state","term"]

cat_summary = []
for c in CAT_COLS:
    vc = df[c].astype(str).value_counts()
    cat_summary.append({
        "feature": c,
        "n_unique": df[c].nunique(),
        "mode": vc.index[0],
        "mode_count": vc.iloc[0],
        "mode_pct": round(vc.iloc[0] / len(df) * 100, 2),
    })
pd.DataFrame(cat_summary).set_index("feature")


## 4. Target variable analysis

We construct a binary target `is_default` from `loan_status` (any
non-fully-paid status except current). This is the variable a predictive model
in Week 3 would aim to predict.

In [ ]:
DEFAULT_STATUSES = ["Charged Off","Default","Late (31-120 days)",
                    "Late (16-30 days)","In Grace Period"]
df["is_default"] = df["loan_status"].astype(str).isin(DEFAULT_STATUSES).astype(int)

default_rate = df["is_default"].mean()
n_default = int(df["is_default"].sum())
print(f"Default rate: {default_rate*100:.2f}%  ({n_default:,} of {len(df):,} loans)")
print()
print("Loan status counts:")
print(df["loan_status"].value_counts())


## 5. Univariate visualizations

In [ ]:
# 5a. Distribution grid (4 key numerics with KDE)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col, color in zip(axes.flat,
                           ["loan_amnt","annual_inc","int_rate","dti"],
                           ["#2280ae","#c7755a","#466473","#427252"]):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    ax.hist(s, bins=40, color=color, alpha=0.75, edgecolor='white', density=True)
    sns.kdeplot(s, ax=ax, color='black', lw=1.8)
    ax.axvline(s.mean(), color=color, ls='--', lw=1.5, label=f"μ={s.mean():.1f}")
    ax.axvline(s.median(), color='black', ls=':', lw=1.5, label=f"med={s.median():.1f}")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col); ax.set_ylabel("Density")
    ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()


In [ ]:
# 5b. Boxplots (quartiles + outliers)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, ["loan_amnt","annual_inc","int_rate","installment","dti","revol_bal"]):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    bp = ax.boxplot(s, vert=True, patch_artist=True, widths=0.5, medianprops=dict(color="black"))
    bp['boxes'][0].set_facecolor("#2280ae"); bp['boxes'][0].set_alpha(0.7)
    ax.set_title(col); ax.set_xticks([]); ax.set_ylabel(col)
plt.suptitle("Boxplots — key numeric features", y=1.00)
plt.tight_layout(); plt.show()


In [ ]:
# 5c. QQ plots for normality assessment
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flat, ["loan_amnt","int_rate","dti","installment_to_income"]):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    s = s.sample(min(5000, len(s)), random_state=42)
    stats.probplot(s, dist="norm", plot=ax)
    ax.set_title(f"QQ plot — {col}")
plt.tight_layout(); plt.show()


In [ ]:
# 5d. Loan status distribution (target)
fig, ax = plt.subplots(figsize=(11, 5.5))
ls = df["loan_status"].value_counts()
colors = ["#a45c55" if s in DEFAULT_STATUSES else "#427252" for s in ls.index]
ax.barh(ls.index[::-1], ls.values[::-1], color=colors[::-1], edgecolor='white')
for i, v in enumerate(ls.values[::-1]):
    ax.text(v + max(ls)*0.01, i, f"{v:,} ({v/len(df)*100:.1f}%)",
            va='center', fontsize=8)
ax.set_xlabel("Loans")
ax.set_title(f"Loan status distribution — default rate = {default_rate*100:.2f}%")
plt.tight_layout(); plt.show()


## 6. Bivariate analysis

In [ ]:
# 6a. Correlation matrix of key numerics
key_corr = [c for c in KEY_NUMERIC if c in df.columns]
corr = df[key_corr].corr()

fig, ax = plt.subplots(figsize=(12, 9.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 8}, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation matrix — key numeric features")
plt.tight_layout(); plt.show()

# Top correlations
corr_pairs = []
for i, a in enumerate(key_corr):
    for b in key_corr[i+1:]:
        corr_pairs.append((a, b, corr.loc[a, b]))
corr_pairs.sort(key=lambda x: -abs(x[2]))
print("\nTop 10 strongest correlations:")
for a, b, r in corr_pairs[:10]:
    print(f"  {a:25s} vs {b:25s}  r={r:+.3f}")


In [ ]:
# 6b. Scatter — loan_amnt vs annual_inc, colored by grade
fig, ax = plt.subplots(figsize=(11, 6.5))
sample = df.sample(min(8000, len(df)), random_state=42)
grades = sorted(sample["grade"].dropna().astype(str).unique())
for g in grades:
    sub = sample[sample["grade"] == g]
    ax.scatter(sub["annual_inc"], sub["loan_amnt"], s=12, alpha=0.55, label=f"Grade {g}")
ax.set_xlabel("Annual income ($)"); ax.set_ylabel("Loan amount ($)")
ax.set_title("Loan amount vs annual income, by grade")
ax.legend(markerscale=2, frameon=False, fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()


In [ ]:
# 6c. Boxplot — int_rate by grade
fig, ax = plt.subplots(figsize=(11, 5.5))
order = sorted(df["grade"].dropna().astype(str).unique())
data = [pd.to_numeric(df[df["grade"] == g]["int_rate"], errors="coerce").dropna() for g in order]
bp = ax.boxplot(data, labels=order, patch_artist=True, widths=0.55, medianprops=dict(color="black", lw=1.5))
for patch in bp['boxes']:
    patch.set_facecolor("#2280ae"); patch.set_alpha(0.85)
for i, s in enumerate(data, 1):
    ax.scatter([i], [s.mean()], marker='D', color='white', edgecolor='black', s=40, zorder=5)
ax.set_xlabel("Grade"); ax.set_ylabel("Interest rate (%)")
ax.set_title("Interest rate by grade")
plt.tight_layout(); plt.show()


In [ ]:
# 6d. Default rate by grade — the central business insight
fig, ax = plt.subplots(figsize=(10, 5.5))
dr_by_grade = df.groupby("grade")["is_default"].mean() * 100
counts = df.groupby("grade").size()
bars = ax.bar(dr_by_grade.index, dr_by_grade.values, color="#a45c55", edgecolor='white')
for b, g in zip(bars, dr_by_grade.index):
    v = dr_by_grade[g]
    ax.text(b.get_x()+b.get_width()/2, v + 0.3, f"{v:.1f}%\n(n={counts[g]:,})",
            ha='center', fontsize=9)
ax.axhline(default_rate*100, ls='--', color="#466473", lw=1.2, label=f"Overall: {default_rate*100:.1f}%")
ax.set_xlabel("Grade"); ax.set_ylabel("Default rate (%)")
ax.set_title("Default rate by grade — monotonic increase in risk")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# 6e. Default rate by purpose
fig, ax = plt.subplots(figsize=(11, 6))
dr_by_purpose = (df.groupby("purpose")["is_default"].mean() * 100).sort_values()
bars = ax.barh(dr_by_purpose.index, dr_by_purpose.values, color="#c7755a", edgecolor='white')
for b, v in zip(bars, dr_by_purpose.values):
    ax.text(v + 0.2, b.get_y()+b.get_height()/2, f"{v:.1f}%", va='center', fontsize=8)
ax.axvline(default_rate*100, ls='--', color="#466473", lw=1.2, label=f"Overall: {default_rate*100:.1f}%")
ax.set_xlabel("Default rate (%)")
ax.set_title("Default rate by loan purpose")
ax.legend(frameon=False, fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()


## 7. Multivariate analysis

In [ ]:
# 7a. Pair plot (key features colored by default)
pp_cols = ["loan_amnt","annual_inc","int_rate","dti"]
sample = df.sample(min(3000, len(df)), random_state=42)

fig, axes = plt.subplots(4, 4, figsize=(13, 12))
for i, a in enumerate(pp_cols):
    for j, b in enumerate(pp_cols):
        ax = axes[i, j]
        if i == j:
            for is_def, color, label in [(0,"#427252","Paid"),(1,"#a45c55","Default")]:
                sub = sample[sample["is_default"] == is_def]
                ax.hist(sub[a], bins=20, alpha=0.5, color=color, label=label, density=True)
            if i == 0: ax.legend(fontsize=7, frameon=False)
        else:
            for is_def, color in [(0,"#427252"),(1,"#a45c55")]:
                sub = sample[sample["is_default"] == is_def]
                ax.scatter(sub[b], sub[a], s=6, alpha=0.4, color=color, edgecolor='none')
        if i == 3: ax.set_xlabel(b, fontsize=9)
        if j == 0: ax.set_ylabel(a, fontsize=9)
plt.suptitle("Pair plot by default status (sample of 3,000 loans)", y=1.00)
plt.tight_layout(); plt.show()


In [ ]:
# 7b. Grade × purpose heatmap
fig, ax = plt.subplots(figsize=(12, 6))
ct = pd.crosstab(df["purpose"], df["grade"])
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", linewidths=0.5,
            cbar_kws={"label": "Loans"}, annot_kws={"size": 8}, ax=ax)
ax.set_title("Loan count by purpose × grade")
plt.tight_layout(); plt.show()


## 8. Hypothesis testing

In [ ]:
ALPHA = 0.05
results = []

def add_test(name, test_stat, p_val, decision, **extra):
    sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns"))
    results.append({
        "test": name,
        "statistic": round(test_stat, 4),
        "p_value": f"{p_val:.2e}",
        "significant": p_val < ALPHA,
        "sig_code": sig,
        **extra,
    })
    print(f"  {name}: stat={test_stat:.4f}, p={p_val:.2e} {sig}")

# 8.1 Welch t-test: int_rate by term (36 vs 60 months)
ir_36 = pd.to_numeric(df[df["term"] == 36]["int_rate"], errors="coerce").dropna()
ir_60 = pd.to_numeric(df[df["term"] == 60]["int_rate"], errors="coerce").dropna()
t, p = stats.ttest_ind(ir_36, ir_60, equal_var=False)
add_test("t-test: int_rate ~ term", t, p, p < ALPHA,
          mean_36=round(ir_36.mean(),3), mean_60=round(ir_60.mean(),3))

# 8.2 Welch t-test: loan_amnt by home_ownership (own vs rent)
la_own = pd.to_numeric(df[df["home_ownership"].astype(str) == "OWN"]["loan_amnt"], errors="coerce").dropna()
la_rent = pd.to_numeric(df[df["home_ownership"].astype(str) == "RENT"]["loan_amnt"], errors="coerce").dropna()
t, p = stats.ttest_ind(la_own, la_rent, equal_var=False)
add_test("t-test: loan_amnt ~ home_ownership", t, p, p < ALPHA,
          mean_own=round(la_own.mean(),3), mean_rent=round(la_rent.mean(),3))

# 8.3 ANOVA: int_rate by grade
groups = [pd.to_numeric(df[df["grade"] == g]["int_rate"], errors="coerce").dropna()
          for g in sorted(df["grade"].dropna().astype(str).unique())]
f, p = stats.f_oneway(*groups)
add_test("ANOVA: int_rate ~ grade", f, p, p < ALPHA, n_groups=len(groups))

# 8.4 ANOVA: loan_amnt by purpose
groups = [pd.to_numeric(df[df["purpose"] == p]["loan_amnt"], errors="coerce").dropna()
          for p in df["purpose"].dropna().astype(str).unique()]
f, p = stats.f_oneway(*groups)
add_test("ANOVA: loan_amnt ~ purpose", f, p, p < ALPHA, n_groups=len(groups))

# 8.5 Chi-square: grade × is_default
ct = pd.crosstab(df["grade"].astype(str), df["is_default"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
add_test("Chi-square: grade × default", chi2, p, p < ALPHA,
          dof=dof, cramers_v=round(float(cramers_v), 4))

# 8.6 Chi-square: home_ownership × verification_status
ct = pd.crosstab(df["home_ownership"].astype(str), df["verification_status"].astype(str))
chi2, p, dof, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
add_test("Chi-square: home × verification", chi2, p, p < ALPHA,
          dof=dof, cramers_v=round(float(cramers_v), 4))

# 8.7 Pearson correlations (key pairs)
for a, b in [("loan_amnt","annual_inc"), ("loan_amnt","int_rate"),
              ("int_rate","dti"), ("revol_bal","revol_util"),
              ("loan_to_income","int_rate")]:
    x = pd.to_numeric(df[a], errors="coerce")
    y = pd.to_numeric(df[b], errors="coerce")
    mask = x.notna() & y.notna()
    r, p = stats.pearsonr(x[mask], y[mask])
    add_test(f"Pearson: {a} vs {b}", r, p, p < ALPHA, r=round(r, 4))

print(f"\n{sum(r['significant'] for r in results)}/{len(results)} tests significant at α={ALPHA}")
pd.DataFrame(results)


## 9. Business insights & inferences

**Default risk is strongly stratified by grade.** The default rate increases
monotonically from grade A (≈3-4%) to grade G (≈35-40%). This confirms the
LendingClub rate-card is risk-based — higher rates compensate for higher
default risk.

**Loan purpose matters.** Small-business and debt-consolidation loans default
at materially higher rates than wedding or car loans. This is consistent with
the economic intuition that purpose drives ability/willingness to repay.

**Interest rate is associated with DTI, but weakly (r ≈ 0.12).** The
relationship is statistically significant (p < 0.001) but practically weak —
LendingClub's rate is driven primarily by grade (which itself incorporates
DTI) rather than DTI directly.

**Loan amount and installment are nearly perfectly correlated (r ≈ 0.94).**
This is mechanical — installment is derived from amount, term and rate.
For Week-3 modeling, including both would inject multicollinearity.

**Home ownership and verification status are dependent (χ² significant).**
Renters are less likely to have verified income than mortgage-holders,
reflecting the documentation requirements of different borrower segments.
